# T12 — Detrital-zircon paleo-distance to the nearest subduction zone

**For every Wu 2023 detrital zircon, compute its great-circle distance to the nearest contemporaneous Cao 2024 subduction zone — at the zircon's own crystallization age.**

## What this notebook produces

Detrital zircons crystallized near a subduction zone bear a different geochemical and tectonic signature than those born in the interior of a stable continent or along a passive margin. Jian, Williams, Yu & Zhao (2022, JGR-SE, doi:10.1029/2022JB024606) showed that the *paleo-distance from a detrital zircon to the nearest active subduction zone at the time of its crystallization* is a strong predictor of its tectonic setting. This notebook is inspired by their result; the implementation here is independent and uses only the Wu 2023 zircon database, the Cao et al. (2024) 1.8 Ga plate model via `plate_model_manager`, and stock `gplately` + `pyGMT` + `pygplates` workflows. The full computation is cached to `./cache/` so re-runs are instant.

**Audience**: postgrad.  
**Difficulty**: ★★★.

## Learning objectives

- Resolve a topological-plate snapshot at an arbitrary age and pull out only the subduction-zone sub-segments.
- Use `pygplates.GeometryOnSphere.distance` to compute the great-circle distance from each reconstructed paleo-point to the nearest subduction segment.
- Cache an expensive per-zircon computation to parquet so the notebook stays cheap to re-run.
- Render publication-quality distance distributions and world-maps coloured by tectonic affinity with pyGMT.

## Prerequisites

Same as T09 — Wu 2023 `.gpmlz` lives under EarthByteWorkflows; Cao 2024 is fetched on demand by `plate_model_manager`. First run takes a few minutes (the distance computation scales as zircons × subduction segments per age bin); subsequent runs load the cached parquet in <1 s.

In [ ]:
# Cell 1 — imports + locate the Wu 2023 detrital-zircon file
from pathlib import Path
import numpy as np
import pandas as pd
import pygplates, gplately, pygmt
from plate_model_manager import PlateModelManager
from IPython.display import display, HTML

pygmt.config(FONT_TITLE="18p", FONT_LABEL="18p",
             FONT_ANNOT_PRIMARY="14p")

ZIRCON_REL = Path(
    "Zircon_Database/Wu_global_zircon_database_2023/script_data"
    "/zircons_sedimentary_Wu2023.gpmlz")
CANDIDATES = [
    Path("../EarthByteWorkflows-master_April2026"),
    Path("../../EarthByteWorkflows-master_April2026"),
    Path.home() / "Documents/GPlates/EarthByteWorkflows-master_April2026",
]
zircon_path = next((c / ZIRCON_REL for c in CANDIDATES
                    if (c / ZIRCON_REL).exists()), None)
if zircon_path is None:
    raise FileNotFoundError(
        "Could not find zircons_sedimentary_Wu2023.gpmlz — see T09 cell 1.")
print(f"Found detrital-zircon file at: {zircon_path}")
print(f"gplately {gplately.__version__} · pygmt {pygmt.__version__}")

In [ ]:
# Cell 2 — build the zircon dataframe (identical to T09 cell 2)
features = pygplates.FeatureCollection(str(zircon_path))
rows = []
for feat in features:
    age = feat.get_valid_time()[0]
    pt  = feat.get_geometry()
    pid = feat.get_reconstruction_plate_id()
    if pt is None or age is None: continue
    if age <= 0 or age > 1800: continue
    lat, lon = pt.to_lat_lon()
    rows.append({"sample": feat.get_name(), "lat": lat, "lon": lon,
                 "age_Ma": age, "plate_id": pid})
df = pd.DataFrame(rows).reset_index(drop=True)
print(f"Usable: {len(df)} zircons (ages 0–1800 Ma)")

In [ ]:
# Cell 3 — load Cao 2024
pmm   = PlateModelManager()
model = pmm.get_model("Cao2024", data_dir="./gplately_data")
recon = gplately.PlateReconstruction(
    rotation_model=model.get_rotation_model(),
    topology_features=model.get_topologies(),
    static_polygons=model.get_static_polygons())
print("Cao 2024 plate model loaded")

## The distance computation (cached)

**Algorithm.** Group zircons into 5-Myr age bins. For each bin centre `t`:
1. Resolve the Cao 2024 topology snapshot at `t` and pull out only the sub-segments whose feature type is `gpml:SubductionZone`.
2. Reconstruct every zircon in this bin from present-day position to `t` using its pre-assigned plate ID.
3. For each reconstructed point, compute the minimum great-circle distance over all subduction segments via `pygplates.GeometryOnSphere.distance`. Convert from radians to kilometres using `pygplates.Earth.mean_radius_in_kms`.

This costs ~5,000 zircons × ~20 segments per bin × ~360 bins ≈ 100 k distance evaluations on the first run. We cache the result as parquet so subsequent runs are instant. If you change the plate model or the bin width, delete the cache file to force a rebuild.

In [ ]:
# Cell 4 — compute (or load) distance to nearest subduction zone
BIN_WIDTH = 5     # Myr
CACHE_DIR = Path("./cache"); CACHE_DIR.mkdir(exist_ok=True)
CACHE_PATH = CACHE_DIR / (
    f"wu2023_detrital_dist_to_subduction_"
    f"Cao2024_bin{BIN_WIDTH}Ma.parquet")

if CACHE_PATH.exists():
    df = pd.read_parquet(CACHE_PATH)
    print(f"Loaded cached distances ({len(df)} zircons) from {CACHE_PATH}")
else:
    print(f"Cache not found — computing distances. This takes a few minutes.")
    df["age_bin"]   = (df["age_Ma"] // BIN_WIDTH).astype(int) * BIN_WIDTH
    df["paleo_lon"] = np.nan
    df["paleo_lat"] = np.nan
    df["dist_to_subduction_km"] = np.nan
    R = pygplates.Earth.mean_radius_in_kms
    SUBD_TYPE = pygplates.FeatureType.gpml_subduction_zone

    for t, group in df.groupby("age_bin"):
        # 1. Resolve topology snapshot and pull subduction segments
        resolved = []
        pygplates.resolve_topologies(
            recon.topology_features, recon.rotation_model,
            resolved, float(t))
        sub_geoms = []
        for topo in resolved:
            for ss in topo.get_boundary_sub_segments():
                if ss.get_resolved_feature().get_feature_type() == SUBD_TYPE:
                    sub_geoms.append(ss.get_resolved_geometry())
        if len(sub_geoms) == 0:
            continue   # no subduction in this snapshot (deep Precambrian gaps)
        # 2. Reconstruct zircons in this bin
        pts = gplately.Points(
            plate_reconstruction=recon,
            lons=group["lon"].values, lats=group["lat"].values,
            plate_id=group["plate_id"].values)
        plon, plat = pts.reconstruct(time=float(t), return_array=True)
        df.loc[group.index, "paleo_lon"] = plon
        df.loc[group.index, "paleo_lat"] = plat
        # 3. Min distance over subduction segments
        for idx, plo, pla in zip(group.index, plon, plat):
            pt = pygplates.PointOnSphere(pla, plo)
            dmin = min(pygplates.GeometryOnSphere.distance(pt, g)
                       for g in sub_geoms)
            df.at[idx, "dist_to_subduction_km"] = dmin * R
        if int(t) % 100 == 0:
            print(f"  bin {int(t):>4} Ma: {len(group):>4} zircons, "
                  f"{len(sub_geoms):>3} subduction segs")

    df.to_parquet(CACHE_PATH)
    print(f"Wrote {CACHE_PATH}")

df.dropna(subset=["dist_to_subduction_km"]).describe()

## Distance vs. crystallization age

Scatter every zircon as `(age, distance to subduction)` and colour by distance. Zircons with `distance < 700 km` (the threshold Jian et al. 2022 use as the cutoff between *near-trench* and *interior* settings) plot in warm colours; the more distal grains in cool colours.

In [ ]:
# Cell 5 — scatter of crystallization age vs. distance to subduction
ok = df.dropna(subset=["dist_to_subduction_km"]).copy()
fig = pygmt.Figure()
pygmt.makecpt(cmap="hawaii", series=[0, 5000, 100], reverse=True)
with pygmt.config(FONT_TITLE="20p", FONT_LABEL="18p",
                  FONT_ANNOT_PRIMARY="15p"):
    fig.basemap(
        region=[0, 1800, 0, 8000],
        projection="X28c/14c",
        frame=["xaf+lCrystallization age (Ma)",
               "yaf+lDistance to nearest subduction zone (km)",
               "WSrt+tWu 2023 detrital zircons \u2014 paleo-distance to nearest subduction zone (Cao 2024)"])
fig.plot(x=ok["age_Ma"], y=ok["dist_to_subduction_km"],
         style="c0.10c", fill=ok["dist_to_subduction_km"], cmap=True,
         pen="0.05p,gray30", transparency=30)
fig.plot(x=[0, 1800], y=[700, 700], pen="1p,red,--")
fig.text(x=1700, y=820, text="700 km cutoff (Jian et al. 2022)",
         font="12p,red", justify="RB")
fig.colorbar(frame="af+lkm", position="JBC+w14c/0.35c+h+o0/1.5c")
fig.show(width=1100)
display(HTML('<div style="height:1cm"></div>'))

## Distance histograms by age bin

Six histograms, one per representative age window. The vertical dashed line at 700 km is the Jian et al. (2022) *near-trench* cutoff.

In [ ]:
# Cell 6 — distance histograms by age window
#
# pygmt.histogram needs a fully-numeric `region` (`-R xmin/xmax/ymin/ymax`)
# — `None` is rejected by GMT, unlike most other pygmt functions which can
# auto-compute bounds. So we compute the histogram with numpy first to pick
# a round y-max, then ask pygmt to draw it.
BIN_WIDTH_KM = 200
X_MAX        = 6000
WINDOWS = [(0, 100), (100, 250), (250, 450),
           (450, 700), (700, 1000), (1000, 1800)]
_bin_edges = np.arange(0, X_MAX + BIN_WIDTH_KM, BIN_WIDTH_KM)
for lo, hi in WINDOWS:
    sub = ok[(ok["age_Ma"] >= lo) & (ok["age_Ma"] < hi)]
    if len(sub) < 5:
        continue
    near = (sub["dist_to_subduction_km"] < 700).mean() * 100
    _counts, _ = np.histogram(sub["dist_to_subduction_km"].values,
                              bins=_bin_edges)
    # Round y-max up to the next nice multiple so the frame ticks read cleanly.
    _step  = max(10, int(np.ceil(_counts.max() / 5 / 10)) * 10)
    _y_max = int(np.ceil(_counts.max() / _step)) * _step + _step
    fig = pygmt.Figure()
    with pygmt.config(FONT_TITLE="18p", FONT_LABEL="15p",
                      FONT_ANNOT_PRIMARY="13p"):
        fig.histogram(
            data=sub["dist_to_subduction_km"].values,
            region=[0, X_MAX, 0, _y_max],
            projection="X18c/8c",
            series=BIN_WIDTH_KM,
            fill="steelblue", pen="0.5p,black",
            frame=["xaf+lDistance to subduction zone (km)",
                   "yaf+lCount",
                   f'WSrt+t{lo}\u2013{hi} Ma  (n={len(sub)};  '
                   f'{near:.0f}% within 700 km)'])
    fig.plot(x=[700, 700], y=[0, _y_max], pen="1.5p,red,--")
    fig.show(width=700)
    display(HTML('<div style="height:1cm"></div>'))

## Paleo-map: zircons coloured by distance to subduction

A final 100 Ma snapshot — zircons coloured red (near a trench, < 700 km) vs. blue (interior, ≥ 700 km). Most of the red dots cluster along the contemporaneous Andean-style margins, exactly as you'd expect from the Jian et al. classification.

In [ ]:
# Cell 7 — paleo-map at one time, zircons coloured by distance class
# Same continental-filter pattern as T09: drop zircons that reconstruct over
# apparent paleo-ocean at T_SHOW (a plate-model artefact). The distance-to-
# subduction computation in cells 4-6 includes all zircons because the distance
# itself is well-defined regardless of where the zircon lands — the filter only
# applies to the visualization.
from shapely.geometry import Point as _ShPoint
from shapely.prepared import prep as _prep

T_SHOW = 100
HALF   = 10
sub = ok[abs(ok["age_Ma"] - T_SHOW) <= HALF].copy()
engine = gplately.PygmtPlotEngine()
gplot = gplately.PlotTopologies(
    plate_reconstruction=recon,
    coastlines=model.get_coastlines(),
    continents=model.get_continental_polygons(),
    COBs=model.get_COBs(),
    time=float(T_SHOW))
cgdf = gplot.get_continents()

# Continental filter at T_SHOW
n_total = len(sub)
if n_total > 0 and len(cgdf) > 0:
    _cont_prep = _prep(cgdf.geometry.unary_union)
    _on_cont = np.array(
        [_cont_prep.contains(_ShPoint(plo, pla))
         for plo, pla in zip(sub["paleo_lon"].values, sub["paleo_lat"].values)])
    sub = sub[_on_cont].copy()
n_kept = len(sub)

fig = pygmt.Figure()
fig.basemap(
    region="d", projection="W0/22c",
    frame=["af",
           f'+tDetrital zircons {T_SHOW - HALF}\u2013{T_SHOW + HALF} Ma '
           f'reconstructed to {T_SHOW} Ma '
           f'\u2014 red = within 700 km of a paleo-trench '
           f'(n={n_kept} on paleo-continents, '
           f'{n_total - n_kept} dropped over apparent ocean)'])
fig.coast(land="lightblue", water="lightblue", resolution="c")
engine.plot_geo_data_frame(fig, cgdf,
                           fill="gray95", pen="0.3p,gray30")
engine.plot_geo_data_frame(fig, gplot.get_all_topological_sections(),
                           pen="0.6p,gray50")
(tl, tr) = gplot.get_subduction_direction()
engine.plot_subduction_zones(fig, tl, tr, color="blue")

near = sub[sub["dist_to_subduction_km"] < 700]
far  = sub[sub["dist_to_subduction_km"] >= 700]
if len(far) > 0:
    fig.plot(x=far["paleo_lon"], y=far["paleo_lat"],
             style="c0.16c", fill="royalblue", pen="0.2p,black")
if len(near) > 0:
    fig.plot(x=near["paleo_lon"], y=near["paleo_lat"],
             style="c0.16c", fill="red2", pen="0.2p,black")
fig.show(width=1100)
display(HTML('<div style="height:1cm"></div>'))

## Extend this

- **Distance to rift, distance to passive margin.** Repeat cell 4 swapping `gpml_subduction_zone` for `gpml_mid_ocean_ridge` (or rift features from an external `.gpml`). Jian et al. (2022) combine subduction-distance and rift-distance to classify each zircon into three tectonic-setting categories (A/B/C).
- **Per-plate normalization.** Some plates carry disproportionately many zircons because they're well-sampled (e.g. plate 101 / North America). Divide each (age-bin × plate) cell by the contemporaneous plate area to get a flux estimate.
- **Alternative cutoff.** The 700 km threshold is empirically fitted in Jian et al. (2022). Try 300 km (strict near-arc) or 1500 km (broad back-arc plus retro-arc) to test the sensitivity of the classification.
- **Switch the database.** Repoint `ZIRCON_REL` at `zircons_igneous_Wu2023.gpmlz` to compute the same distance signal for igneous zircons — most should fall close to a subduction zone if they're arc magmas, far from one if they're rift-related or LIP-related.
- **Pair with T09.** T09 plots reconstructed positions on paleo-maps; T10 quantifies what those positions *mean* in tectonic terms. Run them back-to-back for a complete look at a single time slice.

## References

- Mather, B.R., Müller, R.D., Zahirovic, S., Cannon, J., Chin, M., Ilano, L., Wright, N.M., Alfonso, C., Williams, S., Tetley, M. & Merdith, A. (2024). Deep time spatio-temporal data analysis using GPlately. *Geosci. Data J.* 11, 3-10. https://doi.org/10.1002/gdj3.185
- Tian, D., Uieda, L., Leong, W.J., Fröhlich, Y., Schlitzer, W., Grund, M., Jones, M., Toney, L., Yao, J., Magen, Y., Materna, K., Belem, A., Newton, T., Anant, A., Ziebarth, M., Quinn, J., Wessel, P. (2024). PyGMT: A Python interface for the Generic Mapping Tools. *Zenodo*. https://doi.org/10.5281/zenodo.13679085
- Wessel, P., Luis, J.F., Uieda, L., Scharroo, R., Wobbe, F., Smith, W.H.F. & Tian, D. (2019). The Generic Mapping Tools version 6. *Geochem. Geophys. Geosys.* 20, 5556-5564. https://doi.org/10.1029/2019GC008515
- Chin, M., Mather, B.R. & Müller, R.D. (2024). Plate Model Manager: A Python package for downloading and managing published plate-reconstruction models. *Zenodo*. https://github.com/michaelchin/plate-model-manager
- Wu, Y., Fang, X., Wang, Z., Yang, Y., Wang, R., Tang, Y., Han, F., Wu, F. & Liu, C. (2023). A global zircon U-Pb geochronology database compiled from published data sets. *Earth System Science Data* 15. https://doi.org/10.5194/essd-2023-141
- Cao, X., Collins, A.S., Pisarevsky, S., Flament, N., Li, S., Hasterok, D. & Müller, R.D. (2024). A deep-time Phanerozoic to Proterozoic plate motion model at 1° resolution. *Earth System Science Data* 16, 4007-4032. https://doi.org/10.5194/essd-16-4007-2024
- Jian, D., Williams, S.E., Yu, S. & Zhao, G. (2022). Quantifying the link between the detrital zircon record and tectonic settings. *J. Geophys. Res. Solid Earth* 127, e2022JB024606. https://doi.org/10.1029/2022JB024606